In [1]:
import sys
import os
import httpx
import json
from typing import Dict, Any, Optional, List
from pydantic import BaseModel, Field
from groq import Groq 
from operator import add

from typing import Literal, Dict, Any, Annotated, List, Optional

In [2]:
client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

In [3]:
class ToolCall(BaseModel):
    name: str
    arguments: dict

class RAGUsedContext(BaseModel):
    id: int
    description: str

class AgentResponse(BaseModel):
    answer: str
    tool_calls: List[ToolCall] = Field(default_factory=list)
    final_answer: bool = Field(default=False)
    retrieved_context_ids: List[RAGUsedContext]

class State(BaseModel):
    # existing fields...
    messages: Annotated[List[Any], add] = []
    answer: str = ""
    iteration: int = Field(default=0)
    final_answer: bool = Field(default=False)
    available_tools: List[Dict[str, Any]] = []
    tool_calls: Optional[List[ToolCall]] = Field(default_factory=list)
    retrieved_context_ids: Annotated[List[RAGUsedContext], add] = []

In [2]:
# FastMCP
from fastmcp import Client

# Flight API

In [4]:
RAPIDAPI_API_KEY = os.environ.get('RAPIDAPI_API_KEY')
RAPIDAPI_BASE_URL = os.environ.get('RAPIDAPI_BASE_URL')

In [7]:
RAPIDAPI_BASE_URL

'google-flights2.p.rapidapi.com'

In [5]:
endpoint = 'searchFlights'
params = {
    "departure_id":"LAX",
    "arrival_id":"JFK",
    "outbound_date":"2025-10-31",
    "travel_class":"ECONOMY",
    "adults":"1",
    "show_hidden":"1",
    "currency":"USD",
    "language_code":"en-US",
    "country_code":"US",
    "search_type":"best"
 }

"""Make a request to the TripAdvisor API"""
# if not RAPIDAPI_API_KEY:
#     return {"error": "TripAdvisor API key not configured. Set RAPIDAPI_API_KEY environment variable."}

# headers = {"accept": "application/json", "key": RAPIDAPI_API_KEY}
headers = {
    "x-rapidapi-host": RAPIDAPI_BASE_URL, 
    "x-rapidapi-key": RAPIDAPI_API_KEY,
    # "content-type":"application/json; charset=utf-8"
    }

if params is None:
    params = {}

# params["key"] = RAPIDAPI_API_KEY

url = f"https://{RAPIDAPI_BASE_URL}/api/v1/{endpoint}"

In [9]:
params

{'departure_id': 'LAX',
 'arrival_id': 'JFK',
 'outbound_date': '2025-10-06',
 'travel_class': 'ECONOMY',
 'adults': '1',
 'show_hidden': '1',
 'currency': 'USD',
 'language_code': 'en-US',
 'country_code': 'US',
 'search_type': 'best'}

In [12]:


async with httpx.AsyncClient() as client:

    response = await client.get(url, headers=headers, params=params, timeout=30.0)
    print(response.raise_for_status())
    print(response.json())
    # return response.json()


# async with httpx.AsyncClient() as client:
#     try:
#         response = await client.get(url, headers=headers, params=params, timeout=30.0)
#         print(response.raise_for_status())
#         print(response.json())
#         # return response.json()
#     except httpx.HTTPStatusError as e:
#         # return {
#         #     "error": f"HTTP error occurred: {e}",
#         #     "details": str(e)
#         # }
#         print("HTTP error occurred")

<Response [200 OK]>
{'status': True, 'message': 'Success', 'timestamp': 1760145587767, 'data': {'itineraries': {'topFlights': [{'departure_time': '31-10-2025 03:20 PM', 'arrival_time': '31-10-2025 11:53 PM', 'duration': {'raw': 333, 'text': '5 hr 33 min'}, 'flights': [{'departure_airport': {'airport_name': 'Los Angeles International Airport', 'airport_code': 'LAX', 'time': '2025-10-31 15:20'}, 'arrival_airport': {'airport_name': 'John F. Kennedy International Airport', 'airport_code': 'JFK', 'time': '2025-10-31 23:53'}, 'duration': {'raw': 333, 'text': '5 hr 33 min'}, 'airline': 'American', 'airline_logo': 'https://www.gstatic.com/flights/airline_logos/70px/AA.png', 'flight_number': 'AA 4', 'aircraft': 'Airbus A321 (Sharklets)', 'seat': 'Average legroom', 'legroom': '31 in', 'extensions': ['Average legroom (31 in)', 'In-seat power & USB outlets', 'Stream media to your device', 'Free Wi-Fi', 'Wi-Fi for a fee', 'Emissions estimate: 556 kg CO2e']}], 'delay': {'values': False, 'text': 0}, 

In [6]:
async def flight_api_request(endpoint:str, params: Dict[str, Any]=None) -> Dict[str, Any]:
    """Make a request to the TripAdvisor API"""
    if not RAPIDAPI_API_KEY:
        return {"error": "TripAdvisor API key not configured. Set RAPIDAPI_API_KEY environment variable."}

    headers = {
        "x-rapidapi-host": RAPIDAPI_BASE_URL, 
        "x-rapidapi-key": RAPIDAPI_API_KEY,
    }

    if params is None:
        params = {}

    url = f"https://{RAPIDAPI_BASE_URL}/api/v1/{endpoint}"

    
    try:
        async with httpx.AsyncClient() as client:
            response = await client.get(url, headers=headers, params=params, timeout=30.0)
            response.raise_for_status()
            data = response.json()
        flights = []
        itineraries = data['data']['itineraries']['topFlights']
        for itinerary in itineraries[:3]:
            flight = itinerary.get('flights')[0]
            flights.append({
                "price": itinerary.get("price"),
                "departure_time": itinerary.get("departure_time"),
                "arrival_time": itinerary.get("arrival_time"),
                "departure_airport": flight.get('departure_airport'),
                "arrival_airport": flight.get('arrival_airport'),
                "duration": flight.get('duration'),
                "airline": flight.get('airline'),
                "airline_logo": flight.get('airline_logo')
            })
        return {"flights": flights}
    except httpx.HTTPStatusError as e:
        return {
            "error": f"HTTP error occurred: {e}",
            "details": str(e)
        }
    

In [7]:
response = await flight_api_request(endpoint, params)

In [8]:
response

{'flights': [{'price': 114,
   'departure_time': '31-10-2025 03:20 PM',
   'arrival_time': '31-10-2025 11:53 PM',
   'departure_airport': {'airport_name': 'Los Angeles International Airport',
    'airport_code': 'LAX',
    'time': '2025-10-31 15:20'},
   'arrival_airport': {'airport_name': 'John F. Kennedy International Airport',
    'airport_code': 'JFK',
    'time': '2025-10-31 23:53'},
   'duration': {'raw': 333, 'text': '5 hr 33 min'},
   'airline': 'American',
   'airline_logo': 'https://www.gstatic.com/flights/airline_logos/70px/AA.png'},
  {'price': 122,
   'departure_time': '31-10-2025 08:52 PM',
   'arrival_time': '01-11-2025 05:30 AM',
   'departure_airport': {'airport_name': 'Los Angeles International Airport',
    'airport_code': 'LAX',
    'time': '2025-10-31 20:52'},
   'arrival_airport': {'airport_name': 'John F. Kennedy International Airport',
    'airport_code': 'JFK',
    'time': '2025-11-1 05:30'},
   'duration': {'raw': 338, 'text': '5 hr 38 min'},
   'airline': 'Fr

In [9]:
def format_flight_results(flight_data, top_n=3):
    flights = flight_data.get("flights", [])[:top_n]
    if not flights:
        return "No flight options found."

    result_lines = []
    for i, flight in enumerate(flights, start=1):
        price = flight.get("price", "N/A")
        airline = flight.get("airline", "Unknown Airline")
        departure_time = flight.get("departure_time", "N/A")
        arrival_time = flight.get("arrival_time", "N/A")
        duration = flight.get("duration", {}).get("text", "N/A")
        
        dep_airport = flight.get("departure_airport", {})
        dep_name = dep_airport.get("airport_name", "Unknown Airport")
        dep_code = dep_airport.get("airport_code", "")
        
        arr_airport = flight.get("arrival_airport", {})
        arr_name = arr_airport.get("airport_name", "Unknown Airport")
        arr_code = arr_airport.get("airport_code", "")

        flight_text = (
            f"{i}. Flight by {airline} from {dep_name} ({dep_code}) "
            f"to {arr_name} ({arr_code})\n"
            f"   - Departure: {departure_time}\n"
            f"   - Arrival: {arrival_time}\n"
            f"   - Duration: {duration}\n"
            f"   - Price: ${price}"
        )
        result_lines.append(flight_text)

    return "\n\n".join(result_lines)

In [10]:
formatted_response = format_flight_results(response, top_n=3)
print(formatted_response)

1. Flight by American from Los Angeles International Airport (LAX) to John F. Kennedy International Airport (JFK)
   - Departure: 31-10-2025 03:20 PM
   - Arrival: 31-10-2025 11:53 PM
   - Duration: 5 hr 33 min
   - Price: $114

2. Flight by Frontier from Los Angeles International Airport (LAX) to John F. Kennedy International Airport (JFK)
   - Departure: 31-10-2025 08:52 PM
   - Arrival: 01-11-2025 05:30 AM
   - Duration: 5 hr 38 min
   - Price: $122

3. Flight by Delta from Los Angeles International Airport (LAX) to John F. Kennedy International Airport (JFK)
   - Departure: 31-10-2025 01:00 PM
   - Arrival: 31-10-2025 09:29 PM
   - Duration: 5 hr 29 min
   - Price: $129


In [37]:
itineraries = response['data']['itineraries']['topFlights']
itineraries[:2]

[{'departure_time': '31-10-2025 03:20 PM',
  'arrival_time': '31-10-2025 11:53 PM',
  'duration': {'raw': 333, 'text': '5 hr 33 min'},
  'flights': [{'departure_airport': {'airport_name': 'Los Angeles International Airport',
     'airport_code': 'LAX',
     'time': '2025-10-31 15:20'},
    'arrival_airport': {'airport_name': 'John F. Kennedy International Airport',
     'airport_code': 'JFK',
     'time': '2025-10-31 23:53'},
    'duration': {'raw': 333, 'text': '5 hr 33 min'},
    'airline': 'American',
    'airline_logo': 'https://www.gstatic.com/flights/airline_logos/70px/AA.png',
    'flight_number': 'AA 4',
    'aircraft': 'Airbus A321 (Sharklets)',
    'seat': 'Average legroom',
    'legroom': '31 in',
    'extensions': ['Average legroom (31 in)',
     'In-seat power & USB outlets',
     'Stream media to your device',
     'Free Wi-Fi',
     'Wi-Fi for a fee',
     'Emissions estimate: 556 kg CO2e']}],
  'delay': {'values': False, 'text': 0},
  'self_transfer': False,
  'layovers

In [38]:
flight_response = []

for itinerary in itineraries[:3]:
    flight = itinerary.get('flights')[0]
    flight_response.append({
        "price": itinerary.get("price"),
        "airline": flight.get('airline')
    })
flight_response

[{'price': 114, 'airline': 'American'},
 {'price': 122, 'airline': 'Frontier'},
 {'price': 124, 'airline': 'Delta'}]

In [13]:
response_json = response.json()

In [19]:
response_json['data']['itineraries'].keys()

dict_keys(['topFlights', 'otherFlights'])

In [20]:
top_flights = response_json['data']['itineraries']['topFlights']

In [22]:
len(top_flights)

4

In [23]:
top_flights[:2]

[{'departure_time': '06-10-2025 08:53 PM',
  'arrival_time': '07-10-2025 05:30 AM',
  'duration': {'raw': 337, 'text': '5 hr 37 min'},
  'flights': [{'departure_airport': {'airport_name': 'Los Angeles International Airport',
     'airport_code': 'LAX',
     'time': '2025-10-6 20:53'},
    'arrival_airport': {'airport_name': 'John F. Kennedy International Airport',
     'airport_code': 'JFK',
     'time': '2025-10-7 05:30'},
    'duration': {'raw': 337, 'text': '5 hr 37 min'},
    'airline': 'Frontier',
    'airline_logo': 'https://www.gstatic.com/flights/airline_logos/70px/F9.png',
    'flight_number': 'F9 2504',
    'aircraft': 'Airbus A321neo',
    'seat': 'Below average legroom',
    'legroom': '28 in',
    'extensions': ['Below average legroom (28 in)',
     'Emissions estimate: 258 kg CO2e']}],
  'delay': {'values': False, 'text': 0},
  'self_transfer': False,
  'layovers': None,
  'bags': {'carry_on': 0, 'checked': 0},
  'carbon_emissions': {'difference_percent': -4,
   'CO2e': 2

# Prompt Engineering

In [ ]:
params = {
    "departure_id":"LAX",
    "arrival_id":"JFK",
    "outbound_date":"2025-10-06",
    "travel_class":"ECONOMY",
    "adults":"1",
    "show_hidden":"1",
    "currency":"USD",
    "language_code":"en-US",
    "country_code":"US",
    "search_type":"best"
 }

In [2]:
from datetime import date

In [33]:
# class FlightPlanner(BaseModel):
#     departure_airport_code: str 
#     arrival_airport_code: str 
#     departure_date: date 
#     travel_class: str = 'ECONOMY'
#     num_adults: int = 1

class FlightPlanner(BaseModel):
    departure_id: str = Field(..., description="The departure airport code IATA codes")
    arrival_id: str = Field(..., description="The arrival airport code IATA codes")
    departure_date: date = Field(..., description="The date of departure in YYYY-MM-DD format")
    travel_class: str = Field('ECONOMY', description="Travel class, e.g. ECONOMY, BUSINESS")
    num_adults: int = Field(1, description="Number of adults traveling")
    

In [4]:
import instructor

client = instructor.from_provider("groq/llama-3.3-70b-versatile")

In [39]:
query = """
1. Departure airport: Los Angeles
2. Arrival airport: New York City
3. departure date: Oct 6, 2025
4. travel class: economy
5. Adults: 1
"""

In [ ]:
flight = client.chat.create_with_completion(
    messages=[
        {"role": "system", 
         "content": """
You are an expert data extraction assistant. Extract the following fields exactly:
- departure_id: departure airport code (e.g., LAX)
- arrival_id: arrival airport code (e.g., JFK)
- departure_date: Date in ISO format YYYY-MM-DD (e.g., 2025-10-06)
- travel_class: Travel class (e.g. ECONOMY)
- num_adults: Number of adults traveling

Ensure the departure_date is extracted and output in the format YYYY-MM-DD regardless of how it appears in the input.

"""},
    {
        "role": "user",
        "content": {query}}
    ],
    response_model=FlightPlanner
)

In [43]:
flight

(FlightPlanner(departure_id='LAX', arrival_id='JFK', departure_date=datetime.date(2025, 10, 6), travel_class='ECONOMY', num_adults=1),
 ChatCompletion(id='chatcmpl-354d775c-6a95-413c-a9f6-0212a7ba8f86', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, role='assistant', executed_tools=None, function_call=None, reasoning=None, tool_calls=[ChatCompletionMessageToolCall(id='pa4ea45ab', function=Function(arguments='{"arrival_id":"JFK","departure_date":"2025-10-06","departure_id":"LAX","num_adults":1,"travel_class":"ECONOMY"}', name='FlightPlanner'), type='function')]))], created=1759707651, model='llama-3.3-70b-versatile', object='chat.completion', system_fingerprint='fp_155ab82e98', usage=CompletionUsage(completion_tokens=49, prompt_tokens=558, total_tokens=607, completion_time=0.094295413, prompt_time=0.046621081, queue_time=0.220579681, total_time=0.140916494), usage_breakdown=None, x_groq={'id': 'req_01k6vbzqdye5rv27dfc2h539

# MCP Prompt

In [4]:
from fastmcp import Client

In [3]:
client = Client("http://flight_mcp_server:8000/mcp")

async with client:
    tools = await client.list_tools()

In [4]:
tools

[Tool(name='get_flights', title=None, description='', inputSchema={'properties': {'params': {'additionalProperties': True, 'default': None, 'title': 'Params', 'type': 'object'}, 'top_k': {'default': 3, 'title': 'top_k', 'type': 'string'}}, 'title': 'get_flightsArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'additionalProperties': True, 'title': 'Result', 'type': 'object'}}, 'required': ['result'], 'title': 'get_flightsOutput', 'type': 'object'}, icons=None, annotations=None, meta=None)]

In [5]:

client = Client("http://flight_mcp_server:8000/mcp")

async def call_get_flights():
    async with client:
        params = {
            "departure_id":"LAX",
            "arrival_id":"JFK",
            "outbound_date":"2025-10-31",
            "travel_class":"ECONOMY",
            "adults":"1",
            "show_hidden":"1",
            "currency":"USD",
            "language_code":"en-US",
            "country_code":"US",
            "search_type":"best"
        }
        top_k = 3
        result = await client.call_tool("get_flights", {"params": params, "top_k": top_k})
        print(result)

results = await call_get_flights()

CallToolResult(content=[TextContent(type='text', text='1. Flight by American from Los Angeles International Airport (LAX) to John F. Kennedy International Airport (JFK)\n   - Departure: 31-10-2025 03:20 PM\n   - Arrival: 31-10-2025 11:53 PM\n   - Duration: 5 hr 33 min\n   - Price: $114\n\n2. Flight by Frontier from Los Angeles International Airport (LAX) to John F. Kennedy International Airport (JFK)\n   - Departure: 31-10-2025 08:52 PM\n   - Arrival: 01-11-2025 05:30 AM\n   - Duration: 5 hr 38 min\n   - Price: $122\n\n3. Flight by Delta from Los Angeles International Airport (LAX) to John F. Kennedy International Airport (JFK)\n   - Departure: 31-10-2025 01:00 PM\n   - Arrival: 31-10-2025 09:29 PM\n   - Duration: 5 hr 29 min\n   - Price: $129', annotations=None, meta=None)], structured_content={'result': '1. Flight by American from Los Angeles International Airport (LAX) to John F. Kennedy International Airport (JFK)\n   - Departure: 31-10-2025 03:20 PM\n   - Arrival: 31-10-2025 11:53